# Day 14: Bigram Language Model — Predict the Next Token

**Goal:** Build the simplest possible language model — and discover it uses the exact same training task as GPT.

### The big idea

A bigram model predicts the next token given **only** the previous one:

```
"the cat sat on the ___"
        ↓
    (bigram sees only "the")
        ↓
    P(mat)=0.4, P(floor)=0.2, P(roof)=0.1, ...
```

It's a terrible model (no context!) — but the **task** of "predict the next token" is identical to what GPT does. We're building GPT's great-grandparent.

### Today's plan

1. Train on Shakespeare-style text, character-level
2. Build a bigram via **counting** (no training needed)
3. Build the same model with a **neural net** (`nn.Embedding(V, V)`)
4. Compare both — they converge to the same answer
5. Generate text by sampling one character at a time
6. Compute **perplexity** — the standard LM metric
7. See exactly why bigrams aren't enough → motivates attention

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

## 1. The Data — A Shakespeare-Like Snippet

Same kind of corpus as Day 4. Character-level so the vocabulary stays small and inspectable.

In [ ]:
# A small Shakespeare-like passage. Larger = better model.

text = """to be or not to be that is the question
whether tis nobler in the mind to suffer
the slings and arrows of outrageous fortune
or to take arms against a sea of troubles
and by opposing end them to die to sleep
no more and by a sleep to say we end
the heart ache and the thousand natural shocks
that flesh is heir to tis a consummation
devoutly to be wished to die to sleep
to sleep perchance to dream ay there's the rub
for in that sleep of death what dreams may come
when we have shuffled off this mortal coil
must give us pause there's the respect
that makes calamity of so long life"""

# Build character vocabulary
chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}

print(f"Corpus length: {len(text)} chars")
print(f"Vocabulary ({vocab_size} chars):")
print(f"  {chars}")

# Encode the whole text as a tensor of integers
data = torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)
print(f"\nEncoded shape: {data.shape}")
print(f"First 30 tokens: {data[:30].tolist()}")
print(f"Decoded:        '{text[:30]}'")

## 2. The Counting Bigram — No Training Needed

For every consecutive pair `(prev, next)`, increment `counts[prev, next]`. Then normalize each row.

This is a probability table without any neural network.

In [ ]:
# Count every bigram in the data

counts = torch.zeros((vocab_size, vocab_size), dtype=torch.float32)

for i in range(len(data) - 1):
    prev = data[i].item()
    nxt = data[i + 1].item()
    counts[prev, nxt] += 1

# Normalize each row to get P(next | prev)
# Add 1 ("Laplace smoothing") so no zero probabilities (avoids -inf in log later)
probs = (counts + 1) / (counts + 1).sum(dim=1, keepdim=True)

print(f"Counts matrix shape: {counts.shape}")
print(f"  Each cell counts['a', 'b'] = how often 'b' followed 'a'\n")

# Show the most common next-characters for a few common chars
for char in ['t', 'h', ' ', 'a']:
    if char in char_to_idx:
        idx = char_to_idx[char]
        row = counts[idx]
        # Top-5 most common followers
        top = torch.topk(row, 5)
        followers = [(idx_to_char[i.item()], int(c.item())) for i, c in zip(top.indices, top.values)]
        display_char = repr(char)
        print(f"Most common chars after {display_char:>4}:")
        for f, count in followers:
            display = repr(f)
            print(f"    {display:>4}: {count}")
        print()

In [ ]:
# Visualize the count matrix as a heatmap

plt.figure(figsize=(10, 8))
plt.imshow(counts.numpy(), cmap='Blues', aspect='auto')
plt.colorbar(label='Bigram count')
plt.xticks(range(vocab_size), chars, fontsize=9)
plt.yticks(range(vocab_size), chars, fontsize=9)
plt.xlabel('Next character')
plt.ylabel('Previous character')
plt.title('Bigram counts — which characters follow which?')
plt.tight_layout()
plt.show()

print("Dark cells = pairs that appear often in our corpus.")
print("Notice the bright vertical band — most things are followed by SPACE.")
print("Pattern: 'th' is very common (English!), 'qu' is rare unless 'q' present, etc.")

### Generate text from the counting model

Just walk through the probability table — at each step, sample the next character.

In [ ]:
def sample_from_counts(probs, start='t', max_len=200):
    """Generate text by walking the probability table."""
    current = char_to_idx[start]
    output = [start]
    for _ in range(max_len):
        # Get the probability row for the current character
        row = probs[current]
        # Sample a next character from this distribution
        next_idx = torch.multinomial(row, num_samples=1).item()
        output.append(idx_to_char[next_idx])
        current = next_idx
    return ''.join(output)

print("--- Generated text from COUNTING bigram ---\n")
torch.manual_seed(0)
print(sample_from_counts(probs, start='t', max_len=300))

### Observation

The output is **garbled but English-shaped** — looks like words could form, has vowels and consonants in roughly the right places. But it's not real English. That's the bigram limit: it only sees ONE previous character.

---

## 3. The Neural Bigram Model

Now let's build the SAME thing as a neural network. This is GPT's actual architecture, just minimal.

### The whole model:
```
input token id → nn.Embedding(V, V) → logits → softmax → probability over next token
```

Yes, that's it. One embedding layer with shape `(vocab_size, vocab_size)`.

In [ ]:
class BigramLM(nn.Module):
    """The simplest possible language model.
    Each row of the embedding IS the logits for what comes next.
    """
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, idx, targets=None):
        # idx: (batch, seq_len)
        logits = self.embedding(idx)              # (batch, seq_len, vocab_size)
        
        loss = None
        if targets is not None:
            # CrossEntropy wants (N, C) and (N,)
            B, T, V = logits.shape
            loss = F.cross_entropy(
                logits.view(B*T, V),
                targets.view(B*T)
            )
        return logits, loss

# Build it
torch.manual_seed(42)
model = BigramLM(vocab_size)
print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  (just vocab_size² = {vocab_size}² = {vocab_size * vocab_size})")

### Build training data — random chunks of the corpus

Same pattern as Day 4: input = a chunk of tokens, target = same chunk shifted by 1.

In [ ]:
# Random-batch sampling

BLOCK_SIZE = 8       # how many tokens in one chunk
BATCH_SIZE = 32      # how many chunks per batch

def get_batch(data, block_size=BLOCK_SIZE, batch_size=BATCH_SIZE):
    """Random chunks of (block_size+1) tokens. x = first block_size, y = shifted by 1."""
    starts = torch.randint(0, len(data) - block_size, (batch_size,))
    x = torch.stack([data[s : s + block_size] for s in starts])
    y = torch.stack([data[s + 1 : s + block_size + 1] for s in starts])
    return x, y

# Try one batch
xb, yb = get_batch(data)
print(f"x batch shape: {xb.shape}    (batch_size × block_size)")
print(f"y batch shape: {yb.shape}\n")

print("First chunk from this batch:")
print(f"  x: '{ ''.join(idx_to_char[i.item()] for i in xb[0]) }'")
print(f"  y: '{ ''.join(idx_to_char[i.item()] for i in yb[0]) }'")
print(f"  → y is x shifted by 1 position")

### Train the model

What loss do we expect?

- **Random initialization** → loss ≈ `log(vocab_size)` (random guessing)
- **After training** → loss should drop substantially

In [ ]:
# Check baseline loss before training (random init)
torch.manual_seed(42)
model = BigramLM(vocab_size)
xb, yb = get_batch(data)
_, init_loss = model(xb, yb)
baseline = np.log(vocab_size)
print(f"Random-init loss:   {init_loss.item():.3f}")
print(f"Expected baseline:  {baseline:.3f}  (= log(vocab_size) for random guessing)")
print(f"Anything substantially lower than {baseline:.3f} means the model is learning.\n")

# Train
optimizer = torch.optim.AdamW(model.parameters(), lr=0.1)
losses = []
for step in range(3000):
    xb, yb = get_batch(data)
    _, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step % 500 == 0:
        print(f"Step {step:4d}: loss={loss.item():.4f}")

print(f"\nFinal loss: {losses[-1]:.4f}  (much better than baseline {baseline:.3f})")

In [ ]:
# Plot training loss with baseline marked
plt.figure(figsize=(10, 4))
plt.plot(losses, 'b-', linewidth=1, alpha=0.7)
# Smoothed version for readability
window = 50
smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
plt.plot(range(window-1, len(losses)), smoothed, 'r-', linewidth=2, label='Smoothed')
plt.axhline(y=baseline, color='gray', linestyle='--', label=f'Random baseline = log({vocab_size}) ≈ {baseline:.2f}')
plt.xlabel('Step')
plt.ylabel('Cross-Entropy Loss')
plt.title('Bigram LM training — comfortably beats random baseline')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Counting vs Neural — Do They Agree?

Both methods should learn essentially the same probability table. Let's verify:

In [ ]:
# Extract probability table from the neural model
with torch.no_grad():
    # Forward pass for every possible token (0..vocab_size-1)
    all_ids = torch.arange(vocab_size).unsqueeze(0)   # (1, vocab_size)
    logits = model.embedding(all_ids)[0]               # (vocab_size, vocab_size)
    neural_probs = F.softmax(logits, dim=1)

# Compare side by side as heatmaps
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].imshow(probs.numpy(), cmap='Blues', aspect='auto')
axes[0].set_title('Counting probabilities (Laplace-smoothed)')
axes[0].set_xticks(range(vocab_size))
axes[0].set_xticklabels(chars, fontsize=8)
axes[0].set_yticks(range(vocab_size))
axes[0].set_yticklabels(chars, fontsize=8)
axes[0].set_xlabel('Next char')
axes[0].set_ylabel('Prev char')

axes[1].imshow(neural_probs.numpy(), cmap='Blues', aspect='auto')
axes[1].set_title('Neural model probabilities (after training)')
axes[1].set_xticks(range(vocab_size))
axes[1].set_xticklabels(chars, fontsize=8)
axes[1].set_yticks(range(vocab_size))
axes[1].set_yticklabels(chars, fontsize=8)
axes[1].set_xlabel('Next char')

plt.tight_layout()
plt.show()

# Numerical agreement
diff = (probs - neural_probs).abs().mean().item()
print(f"Mean absolute difference: {diff:.4f}")
print("They should look almost identical — counting and the neural model converge to the same answer.")

## 5. Generate Text — The Neural Way

Same loop as the counting version, but now using the neural model. This is the EXACT generation procedure used by GPT, BERT, and friends.

In [ ]:
def generate(model, start_token, max_new=200, temperature=1.0):
    """Generate text by repeatedly predicting one token at a time."""
    model.eval()
    idx = torch.tensor([[char_to_idx[start_token]]], dtype=torch.long)  # (1, 1)
    
    output = [start_token]
    with torch.no_grad():
        for _ in range(max_new):
            logits, _ = model(idx)                  # (1, current_len, vocab_size)
            last_logits = logits[0, -1, :] / temperature
            probs = F.softmax(last_logits, dim=0)
            next_idx = torch.multinomial(probs, num_samples=1).item()
            output.append(idx_to_char[next_idx])
            
            # Append the next token to our running input
            idx = torch.cat([idx, torch.tensor([[next_idx]])], dim=1)
    return ''.join(output)

print("--- Generated text from the NEURAL bigram ---\n")
torch.manual_seed(0)
print(generate(model, start_token='t', max_new=300))

print("\n--- Same output, higher temperature (more random) ---\n")
torch.manual_seed(0)
print(generate(model, start_token='t', max_new=300, temperature=2.0))

print("\n--- Lower temperature (more predictable) ---\n")
torch.manual_seed(0)
print(generate(model, start_token='t', max_new=300, temperature=0.5))

## 6. Perplexity — The Standard LM Metric

**Perplexity = exp(loss).** Intuition: "how many tokens does the model effectively have to choose between?"

- Perfect model: perplexity = 1 (knows exactly which token comes next)
- Random model: perplexity = vocab_size
- Real models: somewhere in between (lower = better)

LLM papers always report perplexity. Knowing it lets you compare models.

In [ ]:
# Compute average loss + perplexity over the WHOLE corpus

model.eval()
with torch.no_grad():
    # Run the whole corpus through, get one big loss
    x = data[:-1].unsqueeze(0)              # (1, N-1)
    y = data[1:].unsqueeze(0)                # (1, N-1)
    _, avg_loss = model(x, y)

perplexity = np.exp(avg_loss.item())
random_perplexity = vocab_size

print(f"Average cross-entropy loss: {avg_loss.item():.3f}")
print(f"Perplexity:                 {perplexity:.2f}")
print(f"\nFor context:")
print(f"  Random guessing perplexity: {random_perplexity}")
print(f"  Perfect model perplexity:   1.00")
print(f"\nInterpretation: 'the model is effectively choosing among ~{perplexity:.0f} tokens at each step'")
print(f"That's better than random ({random_perplexity}), but far from perfect (1).")

## 7. Why Bigrams Are Stuck

The model can only see ONE previous character. Let's see this limitation directly:

In [ ]:
# What does the model predict after various contexts?
# All these end in 'e' — but the bigram model treats them identically.

contexts = ['the', 'be', 'sleepe', 'he', 'come', 'wishe']  # all end in 'e'

print("Bigram limitation demo:")
print("All these contexts end in 'e'. The bigram model gives the SAME prediction.\n")

# Show top 5 predictions for 'e' alone
e_idx = char_to_idx['e']
with torch.no_grad():
    logits = model.embedding(torch.tensor([[e_idx]]))[0, 0]
    probs_after_e = F.softmax(logits, dim=0)

top = probs_after_e.topk(5)
print(f"P(next | last char = 'e'):")
for prob, idx in zip(top.values, top.indices):
    c = idx_to_char[idx.item()]
    display = repr(c)
    print(f"  {display:>5}: {prob.item()*100:.1f}%")

print("\nThis is the SAME distribution regardless of full context:")
print("  'the' → next char = same as 'e' alone")
print("  'be'  → next char = same as 'e' alone")
print("  'come'→ next char = same as 'e' alone")
print("\nObviously these contexts have DIFFERENT likely continuations.")
print("Bigrams can't capture this. We need more context → attention (Day 15+).")

---

## Exercises

1. **More data:** Replace the corpus with a longer text (your favorite book chapter, a song lyrics file, etc.). Does loss go lower?

2. **Word-level bigram:** Change tokenization to word-level. Now bigrams predict the next WORD. Try generating — does it sound more like English?

3. **Trigram model:** Build a model that looks at the LAST TWO characters. Hint: vocabulary expands to `vocab_size^2`. How much better does it do?

4. **Compute perplexity on held-out data:** Split the corpus 90/10. Train on 90%, compute perplexity on 10%. Does it match the training perplexity?

5. **The 'th' problem:** What's `P(h | t)` from the counting model? From the neural model? They should match closely.

---

## Key Takeaways

### The bigram language model

```
input: previous token
output: probability distribution over next token

Model: a single nn.Embedding(V, V) — vocab_size² parameters
Train: Cross-Entropy loss on (input, target) pairs from text
Generate: sample one token at a time, feeding output back as input
```

### What this teaches you

- **Next-token prediction** is the universal LM training task
- **Cross-Entropy loss** is the universal LM loss
- **Temperature-controlled sampling** is the universal generation method
- **Perplexity** = `exp(loss)` is the universal LM metric

These EXACT concepts power GPT-4. The bigram model differs only in:
1. Architecture (1 lookup table vs 96 transformer layers)
2. Context (1 previous token vs 128,000)

### Why bigrams fail

```
"the cat sat on the m..."   ← bigram only sees "m"
"the dog ran on the m..."   ← bigram only sees "m"

These should predict DIFFERENT next chars given context,
but bigram outputs the SAME distribution for both.
```

### Phase 3 progression

```
Day 13: RNN — process sequences, but vanishing gradients     ✓
Day 14: Bigram LM — the next-token task                       ← YOU ARE HERE
Day 15: Self-attention — look at multiple tokens, weighted    ← TOMORROW
Day 16: Multi-head attention
Day 17: Positional encoding
Day 18: Attention-based generator
Day 19+: Transformers (it all comes together)
```

**Tomorrow:** Self-attention. The single most important concept in modern AI. The trick that lets one operation look at many tokens simultaneously — and the heart of every transformer.